# DINOv2 + DermaMNIST 28×28 — no class weights

Train and evaluate 6 classification heads on DermaMNIST (28px images), then test generalizability on **MedIMeta skinl\_derm**.

**Runtime**: GPU (T4 or better recommended)

**Stages:**
1. Feature extraction (DINOv2-base, frozen)
2. Train 6 heads (uniform CrossEntropyLoss)
3. Evaluate on DermaMNIST test set
4. Extract external features (MedIMeta skinl\_derm)
5. External evaluation

## 0. Setup

In [ ]:
import subprocess, sys
!nvidia-smi

In [ ]:
!pip install -q medmnist transformers timm scikit-learn matplotlib seaborn medimeta

### Google Drive (optional)
Mount Drive to persist checkpoints and features across sessions. Skip if you don't need persistence.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/dermamnist_project'  # change as needed
BASE = '/content'   # use /content for session-only storage

### Paths

In [ ]:
from pathlib import Path
FEAT_DIR  = Path(BASE) / 'features/cls'
PATCH_DIR = Path(BASE) / 'features/patch'
CKPT_DIR  = Path(BASE) / 'checkpoints_28_noweight'
PLOT_DIR  = Path(BASE) / 'plots'
MEDIMETA_PATH = Path(BASE) / 'MedIMeta'  # upload or mount your MedIMeta data here
for d in [FEAT_DIR, PATCH_DIR, CKPT_DIR, PLOT_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Paths ready.')

## 1. Imports & Device

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from pathlib import Path
import datetime, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler, label_binarize
from medmnist import DermaMNIST, INFO
from torchvision import transforms
from transformers import AutoModel

def log(msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
N_CLASSES  = 7
BATCH_SIZE = 256
CLASS_NAMES = [v for k,v in sorted(INFO["dermamnist"]["label"].items(), key=lambda x: int(x[0]))]
CLASS_SHORT = ["AK","BCC","BKL","DF","MEL","NV","VASC"]
log(f"Device: {DEVICE}")


## 2. Feature Extraction (DINOv2-base, frozen)

In [ ]:
dinov2_transform = transforms.Compose([
    transforms.Resize(28, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def extract(split):
    cls_path   = FEAT_DIR  / f'X_{split}.npy'
    patch_path = PATCH_DIR / f'X_patch_{split}.npy'
    y_path     = FEAT_DIR  / f'y_{split}.npy'
    if cls_path.exists() and patch_path.exists():
        print(f'[{split}] Loading cached...'); return np.load(cls_path), np.load(y_path), np.load(patch_path), np.load(y_path)
    print(f'[{split}] Extracting...')
    backbone = AutoModel.from_pretrained('facebook/dinov2-base').eval().to(DEVICE)
    dataset  = DermaMNIST(split=split, transform=dinov2_transform, size=28, download=True)
    loader   = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    all_cls, all_patch, all_y = [], [], []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            out     = backbone(pixel_values=imgs.to(DEVICE))
            cls     = out.last_hidden_state[:, 0, :].cpu().numpy()
            patches = out.last_hidden_state[:, 1:, :]
            B       = patches.shape[0]
            patches = patches.permute(0,2,1).reshape(B,768,16,16).cpu().numpy()
            all_cls.append(cls.astype(np.float32))
            all_patch.append(patches.astype(np.float16))
            all_y.append(labels.numpy().flatten())
            if (i+1) % 10 == 0: print(f'  batch {i+1}/{len(loader)}')
    del backbone; torch.cuda.empty_cache()
    X_cls = np.concatenate(all_cls); X_patch = np.concatenate(all_patch)
    y     = np.concatenate(all_y)
    np.save(cls_path, X_cls); np.save(patch_path, X_patch); np.save(y_path, y)
    print(f'  [{split}] CLS={X_cls.shape}  patch={X_patch.shape}')
    return X_cls, y, X_patch, y

X_tr_cls, y_tr, X_tr_patch, _ = extract('train')
X_va_cls, y_va, X_va_patch, _ = extract('val')
X_te_cls, y_te, X_te_patch, _ = extract('test')
print(f'train={len(y_tr)}  val={len(y_va)}  test={len(y_te)}')

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_cls)
X_va_s = scaler.transform(X_va_cls)
X_te_s = scaler.transform(X_te_cls)

def cls_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)
def patch_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X.astype(np.float32)), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)
def attn_loader(X, y, shuffle=False):
    Xt = torch.tensor(X.astype('float32')).permute(0,2,3,1).reshape(-1,256,768)
    ds = TensorDataset(Xt, torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

tr_cls = cls_loader(X_tr_s, y_tr, shuffle=True); va_cls = cls_loader(X_va_s, y_va); te_cls = cls_loader(X_te_s, y_te)
tr_patch = patch_loader(X_tr_patch, y_tr, shuffle=True); va_patch = patch_loader(X_va_patch, y_va); te_patch = patch_loader(X_te_patch, y_te)
tr_attn = attn_loader(X_tr_patch, y_tr, shuffle=True); va_attn = attn_loader(X_va_patch, y_va); te_attn = attn_loader(X_te_patch, y_te)

## 3. Model Definitions

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),    nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, n))
    def forward(self, x): return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, d, drop=0.3):
        super().__init__()
        self.b = nn.Sequential(nn.Linear(d,d), nn.BatchNorm1d(d), nn.ReLU(),
                                nn.Dropout(drop), nn.Linear(d,d), nn.BatchNorm1d(d))
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(x + self.b(x))

class ResidualMLP(nn.Module):
    def __init__(self, in_dim=768, h=512, n=7, drop=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop))
        self.r1 = ResBlock(h, drop); self.r2 = ResBlock(h, drop)
        self.head = nn.Linear(h, n)
    def forward(self, x): return self.head(self.r2(self.r1(self.proj(x))))

class CNNHead(nn.Module):
    def __init__(self, c=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n))
    def forward(self, x): return self.net(x)

class CNNHeadV2(nn.Module):
    def __init__(self, c=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, n))
    def forward(self, x): return self.net(x)

class AttnPoolHead(nn.Module):
    def __init__(self, pd=768, proj=256, heads=4, layers=2, n=7, drop=0.3):
        super().__init__()
        self.proj    = nn.Linear(pd, proj)
        self.pos_emb = nn.Embedding(256, proj)
        enc = nn.TransformerEncoderLayer(d_model=proj, nhead=heads,
              dim_feedforward=proj*4, dropout=drop, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=layers)
        self.attn = nn.Linear(proj, 1)
        self.cls  = nn.Sequential(nn.LayerNorm(proj), nn.Dropout(drop),
                                   nn.Linear(proj,128), nn.GELU(),
                                   nn.Dropout(drop*0.5), nn.Linear(128, n))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        B, N, _ = x.shape
        pos = torch.arange(N, device=x.device).unsqueeze(0)
        h = self.proj(x) + self.pos_emb(pos)
        h = self.transformer(h)
        w = torch.softmax(self.attn(h), dim=1)
        return self.cls((w * h).sum(dim=1))

class CAM(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Linear(channels, channels//r), nn.ReLU(inplace=True),
            nn.Linear(channels//r, channels))
    def forward(self, x):
        b, c, _, _ = x.size()
        max_out = self.linear(F.adaptive_max_pool2d(x,1).view(b,c)).view(b,c,1,1)
        avg_out = self.linear(F.adaptive_avg_pool2d(x,1).view(b,c)).view(b,c,1,1)
        return torch.sigmoid(max_out + avg_out) * x

class SAM(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        concat = torch.cat([torch.max(x,dim=1,keepdim=True)[0],
                            torch.mean(x,dim=1,keepdim=True)], dim=1)
        return torch.sigmoid(self.conv(concat)) * x

class CBAM(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.cam = CAM(channels, r); self.sam = SAM()
    def forward(self, x): return self.sam(self.cam(x)) + x

class CNNHeadCBAM(nn.Module):
    def __init__(self, c=768, n=7):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(c,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2))
        self.cbam1 = CBAM(256)
        self.conv2 = nn.Sequential(nn.Conv2d(256,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.cbam2 = CBAM(128)
        self.conv3 = nn.Sequential(nn.Conv2d(128,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.head  = nn.Sequential(nn.Flatten(), nn.Dropout(0.4),
                                    nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n))
    def forward(self, x):
        x = self.cbam1(self.conv1(x))
        x = self.cbam2(self.conv2(x))
        return self.head(self.conv3(x))

print("All model classes defined.")


## 4. Training

In [ ]:
def train(model, tr_loader, va_loader, ckpt, epochs=100, lr=3e-4,
          label_smooth=0.0, scheduler="cosine", patience=None,
          warmup=0, clip=0.0, noise=0.0, name="model", class_weight=None):
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smooth,
                               weight=class_weight.to(DEVICE) if class_weight is not None else None)
    sch  = CosineAnnealingLR(opt, T_max=epochs) if scheduler == "cosine" else \
           ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10, min_lr=1e-7)
    best, no_imp = 0.0, 0
    hist = {"tl": [], "vl": [], "va": []}
    for ep in range(1, epochs+1):
        if warmup and ep <= warmup:
            for pg in opt.param_groups: pg["lr"] = lr * ep / warmup
        model.train(); tl = 0.0
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            if noise > 0: X_b = X_b + torch.randn_like(X_b) * noise
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            if clip > 0: nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step(); tl += loss.item()
        tl /= len(tr_loader)
        model.eval(); vl, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                vl += crit(logits, y_b).item()
                correct += (logits.argmax(1)==y_b).sum().item()
                total   += y_b.size(0)
        vl /= len(va_loader); va = correct/total
        if scheduler == "cosine": sch.step()
        elif ep > warmup:         sch.step(va)
        hist["tl"].append(tl); hist["vl"].append(vl); hist["va"].append(va)
        if va > best:
            best = va; torch.save(model.state_dict(), ckpt); no_imp=0; flag="✓"
        else:
            no_imp += 1; flag=""
        if patience and no_imp >= patience:
            print(f"  [{name}] Early stop ep {ep}"); break
        if ep % 10 == 0 or ep == 1:
            print(f"  [{name}] ep{ep:03d} | tr={tl:.4f} vl={vl:.4f} va={va:.4f} {flag}")
    print(f"  [{name}] best val acc: {best:.4f}")
    return hist, best


In [ ]:
class_weights = None  # uniform loss
print('Training without class weights.')

configs = [
    ('MLP',         MLPClassifier(),  te_cls,   tr_cls,   va_cls,
     dict(epochs=100, lr=3e-4, scheduler='cosine', name='MLP')),
    ('Residual MLP',ResidualMLP(),    te_cls,   tr_cls,   va_cls,
     dict(epochs=100, lr=3e-4, scheduler='cosine', name='ResMLP')),
    ('CNN Head',    CNNHead(),        te_patch, tr_patch, va_patch,
     dict(epochs=60,  lr=3e-4, label_smooth=0.1, scheduler='cosine', name='CNN')),
    ('CNN Head v2', CNNHeadV2(),      te_patch, tr_patch, va_patch,
     dict(epochs=200, lr=1e-4, label_smooth=0.1, scheduler='plateau', patience=30, warmup=5, name='CNN-v2')),
    ('CNN+CBAM',    CNNHeadCBAM(),    te_patch, tr_patch, va_patch,
     dict(epochs=200, lr=1e-4, label_smooth=0.1, scheduler='plateau', patience=30, warmup=5, name='CNN-CBAM')),
    ('Attn Pool',   AttnPoolHead(),   te_attn,  tr_attn,  va_attn,
     dict(epochs=150, lr=3e-4, label_smooth=0.05, scheduler='plateau', patience=25, warmup=5, clip=1.0, noise=0.005, name='Attn')),
]

histories = {}; test_loaders = {}
for name, model, te_loader, tr_loader, va_loader, kwargs in configs:
    log(f'--- Training: {name} ---')
    model = model.to(DEVICE)
    ckpt  = CKPT_DIR / f"{name.replace(' ','_')}_best.pt"
    h, _  = train(model, tr_loader, va_loader, ckpt, class_weight=class_weights, **kwargs)
    histories[name] = h; test_loaders[name] = te_loader
log('All training complete!')

### Training Curves

In [ ]:
method_order = ['MLP','Residual MLP','CNN Head','CNN Head v2','CNN+CBAM','Attn Pool']
fig, axes = plt.subplots(2, 6, figsize=(26, 7))
for col, name in enumerate(method_order):
    h = histories[name]; ep = range(1, len(h['tl'])+1)
    axes[0,col].plot(ep, h['tl'], label='train', color='#4C72B0')
    axes[0,col].plot(ep, h['vl'], label='val',   color='#DD8452')
    axes[0,col].set_title(f'{name} — Loss', fontsize=9)
    axes[0,col].legend(fontsize=7)
    best_va = max(h['va'])
    axes[1,col].plot(ep, h['va'], color='#55A868')
    axes[1,col].axhline(best_va, color='red', linestyle='--', label=f'best={best_va:.4f}', linewidth=1)
    axes[1,col].set_title(f'{name} — Val Acc', fontsize=9)
    axes[1,col].legend(fontsize=7)
plt.suptitle('DINOv2 28×28 — Training Curves (no class weights)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR / '28_noweight_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluate on DermaMNIST Test Set

In [ ]:
def evaluate(model, loader, ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    logits_all, y_all = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            logits_all.append(model(X_b.to(DEVICE)).cpu().numpy())
            y_all.extend(y_b.numpy())
    logits  = np.concatenate(logits_all)
    probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_true  = np.array(y_all)
    y_pred  = logits.argmax(1)
    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average="macro")
    f1_wtd  = f1_score(y_true, y_pred, average="weighted")
    auc_mac = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    auc_wtd = roc_auc_score(y_true, probs, multi_class="ovr", average="weighted")
    y_bin   = label_binarize(y_true, classes=list(range(N_CLASSES)))
    per_auc = np.array([roc_auc_score(y_bin[:,i], probs[:,i]) for i in range(N_CLASSES)])
    cm      = confusion_matrix(y_true, y_pred)
    per_rec = (cm.astype(float) / cm.sum(axis=1, keepdims=True)).diagonal()
    return {"acc":acc,"f1_mac":f1_mac,"f1_wtd":f1_wtd,
            "auc_mac":auc_mac,"auc_wtd":auc_wtd,"per_auc":per_auc,"per_rec":per_rec}

def evaluate_external(model, loader, ckpt, n_ext=6):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            logits6 = model(X_b.to(DEVICE))[:, 1:]   # drop actinic keratoses
            all_probs.extend(torch.softmax(logits6, dim=1).cpu().numpy())
            all_labels.extend(y_b.numpy())
    probs  = np.array(all_probs); y_true = np.array(all_labels); y_pred = probs.argmax(1)
    acc    = accuracy_score(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average="macro")
    y_bin  = label_binarize(y_true, classes=list(range(n_ext)))
    auc    = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr")
    return acc, auc, f1_mac, y_true, y_pred


In [ ]:
model_map = {
    'MLP': MLPClassifier(), 'Residual MLP': ResidualMLP(),
    'CNN Head': CNNHead(), 'CNN Head v2': CNNHeadV2(),
    'CNN+CBAM': CNNHeadCBAM(), 'Attn Pool': AttnPoolHead(),
}

results = {}
for name, model in model_map.items():
    model = model.to(DEVICE)
    ckpt  = CKPT_DIR / f"{name.replace(' ','_')}_best.pt"
    r = evaluate(model, test_loaders[name], ckpt)
    results[name] = r
    log(f'[{name}]  acc={r["acc"]:.4f}  macro_auc={r["auc_mac"]:.4f}  macro_f1={r["f1_mac"]:.4f}')

print(f"\n{'='*65}")
print(f"{'Method':<18} {'Acc':>7} {'MacroAUC':>10} {'WtdAUC':>9} {'MacroF1':>9}")
print('-'*65)
for m, r in results.items():
    print(f"{m:<18} {r['acc']:>7.4f} {r['auc_mac']:>10.4f} {r['auc_wtd']:>9.4f} {r['f1_mac']:>9.4f}")
print('='*65)

## 6. External Validation (MedIMeta skinl_derm)

> **Setup**: Upload your MedIMeta dataset to `MEDIMETA_PATH` defined in cell 0.  
> Folder structure needed: `MedIMeta/skinl_derm/annotations.csv` and `MedIMeta/skinl_derm/images/`

> Alternatively mount Google Drive where the data is stored.

In [ ]:
from medimeta import MedIMeta
import pandas as pd

ext_dataset = MedIMeta(MEDIMETA_PATH, "skinl_derm", "Diagnosis")
ann = pd.read_csv(f"{MEDIMETA_PATH}/skinl_derm/annotations.csv")
df  = ann[["filepath","split","Diagnosis"]].copy()

collapse_map = {
    "basal cell carcinoma": "basal cell carcinoma",
    "blue nevus": "melanocytic nevi", "clark nevus": "melanocytic nevi",
    "combined nevus": "melanocytic nevi", "congenital nevus": "melanocytic nevi",
    "dermal nevus": "melanocytic nevi", "recurrent nevus": "melanocytic nevi",
    "reed or spitz nevus": "melanocytic nevi",
    "dermatofibroma": "dermatofibroma",
    "lentigo": "benign keratosis-like lesions",
    "seborrheic keratosis": "benign keratosis-like lesions",
    "melanoma": "melanoma", "vascular lesion": "vascular lesions",
    "melanosis": None, "miscellaneous": None,
}
shared_classes = ["basal cell carcinoma","benign keratosis-like lesions",
                  "dermatofibroma","melanoma","melanocytic nevi","vascular lesions"]
class_to_id = {c: i for i, c in enumerate(shared_classes)}
df["class"]     = df["Diagnosis"].map(collapse_map)
df_ext          = df.dropna(subset=["class"]).copy()
df_ext["label"] = df_ext["class"].map(class_to_id)
print(f"External samples: {len(df_ext)}")
print(df_ext.groupby("class").size())

dinov2_norm  = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
backbone_ext = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(DEVICE)
ext_cls_list, ext_patch_list, ext_y_list = [], [], []
with torch.no_grad():
    for i, idx in enumerate(df_ext.index):
        img_tensor, _ = ext_dataset[idx]
        if img_tensor.shape[-1] != 224:
            img_tensor = torch.nn.functional.interpolate(
                img_tensor.unsqueeze(0), size=224, mode="bicubic", align_corners=False).squeeze(0)
        img_tensor = dinov2_norm(img_tensor).unsqueeze(0).to(DEVICE)
        out = backbone_ext(pixel_values=img_tensor)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        patches = out.last_hidden_state[:, 1:, :]
        patches = patches.permute(0,2,1).reshape(1,768,16,16).cpu().numpy()
        ext_cls_list.append(cls[0].astype(np.float32))
        ext_patch_list.append(patches[0].astype(np.float16))
        ext_y_list.append(df_ext.loc[idx, "label"])
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(df_ext)}")
del backbone_ext; torch.cuda.empty_cache()
X_ext_cls   = np.stack(ext_cls_list)
X_ext_patch = np.stack(ext_patch_list)
y_ext       = np.array(ext_y_list)
log(f"External CLS: {X_ext_cls.shape}  patch: {X_ext_patch.shape}")

X_ext_s          = scaler.transform(X_ext_cls)
ext_cls_loader   = cls_loader(X_ext_s, y_ext)
ext_patch_loader = patch_loader(X_ext_patch, y_ext)
ext_attn_loader  = attn_loader(X_ext_patch, y_ext)
ext_loader_map = {
    "MLP": ext_cls_loader, "Residual MLP": ext_cls_loader,
    "CNN Head": ext_patch_loader, "CNN Head v2": ext_patch_loader,
    "CNN+CBAM": ext_patch_loader, "Attn Pool": ext_attn_loader,
}


In [ ]:
log("=== STAGE 5: External Evaluation (skinl_derm) ===")
print(f"\n{\'=\'*70}")
print(f"{\'Method\':<18} {\'Ext Acc\':>9} {\'Ext MacroAUC\':>14} {\'Ext MacroF1\':>12}")
print("-"*70)
ext_results = {}
for name, model in model_map.items():
    model = model.to(DEVICE)
    ckpt  = CKPT_DIR / f"{name.replace(' ',\'_\')}_best.pt"
    acc, auc, f1, y_true, y_pred = evaluate_external(model, ext_loader_map[name], ckpt)
    ext_results[name] = {"acc": acc, "auc": auc, "f1": f1}
    print(f"{name:<18} {acc:>9.4f} {auc:>14.4f} {f1:>12.4f}")
print("="*70)
best_ext = max(ext_results, key=lambda m: ext_results[m]["auc"])
print(f"\nBest external AUC: {best_ext}")
r = ext_results[best_ext]
print(classification_report(r["y_true"] if "y_true" in r else y_true,
                             r["y_pred"] if "y_pred" in r else y_pred,
                             target_names=shared_classes))


## 7. Save Results

In [ ]:
import zipfile
zip_path = Path(BASE) / 'dermamnist_28_noweight_results.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in PLOT_DIR.glob('28_noweight_*.png'):
        z.write(f, f.name)
    for f in CKPT_DIR.glob('*.pt'):
        z.write(f, 'checkpoints/' + f.name)
print(f'Saved: {zip_path}')

# Download the zip from Colab
# from google.colab import files
# files.download(str(zip_path))